# SCM setup and verification

This notebook installs the SCM and verifies it. The recommended course workflow is Google Colab: open the notebook with the badge below and run every cell in order. No local Python installation or kernel change is needed.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/evanwellmeyer/GCM/blob/main/notebooks/01_setup.ipynb)

In [ ]:
from pathlib import Path
import subprocess
import sys

incolab = 'google.colab' in sys.modules
if incolab:
    root = Path('/content/GCM')
    if not root.exists():
        subprocess.run([
            'git', 'clone', '--depth', '1',
            'https://github.com/evanwellmeyer/GCM.git', str(root),
        ], check=True)
else:
    root = Path.cwd().resolve()
    while root != root.parent and not (root / 'pyproject.toml').exists():
        root = root.parent

if not (root / 'pyproject.toml').exists():
    raise FileNotFoundError('Open this notebook from inside the GCM repository.')
if sys.version_info < (3, 11):
    raise RuntimeError('Python 3.11 or newer is required.')

print('repository:', root)
print('running in Colab:', incolab)
print('python:', sys.executable)
print('version:', sys.version.split()[0])

## Install the model

This cell installs the model, plotting tools, and tests into the current Colab runtime. Re-running it is safe.

In [ ]:
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--quiet', '-e', f'{root}[dev]',
], check=True)
print('SCM installation complete')

## Verify the installation

The following cells check the imports and run the complete SCM test suite.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import torch
import scm

print('kernel python:', sys.executable)
print('scm source:', Path(scm.__file__).resolve())
print('numpy:', np.__version__)
print('pytorch:', torch.__version__)
print('accelerator available:', torch.cuda.is_available())

In [ ]:
import subprocess

result = subprocess.run(
    [sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider'],
    cwd=root,
    text=True,
    capture_output=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError('SCM tests did not pass')

Setup is complete when the test cell reports all tests passing. Open [`02_experiments.ipynb` in Colab](https://colab.research.google.com/github/evanwellmeyer/GCM/blob/main/notebooks/02_experiments.ipynb) and run it from the top. Its setup cell installs the SCM again if Colab starts a fresh runtime.